# 🧠 Model A: Standard Multi-Class CNN Classifier
### Baseline Model Architecture with Softmax Output
This notebook builds and trains a standard Convolutional Neural Network (CNN) for categorical ripeness classification.
- Treats unripe, ripe, and overripe as independent nominal categories.
- Uses **Softmax** activation in the final layer.
- Optimizes **Sparse Categorical Cross-Entropy Loss**.


In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

PROJECT_ROOT = Path("..").resolve()
DATASET_DIR = PROJECT_ROOT / "dataset"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 15


## 1. Load Training and Validation Splits


In [ ]:
def load_split(split_name):
    split_dir = DATASET_DIR / split_name
    images, labels = [], []
    for c_idx, c_name in enumerate(["unripe", "ripe", "overripe"]):
        for img_p in (split_dir / c_name).glob("*.jpg"):
            img = Image.open(img_p).convert("RGB").resize(IMAGE_SIZE)
            images.append(np.array(img, dtype=np.float32) / 255.0)
            labels.append(c_idx)
    return np.array(images), np.array(labels)

X_train, y_train = load_split("train")
X_val, y_val = load_split("validation")
print(f"Train Shape: {X_train.shape}, Labels: {y_train.shape}")
print(f"Val Shape: {X_val.shape}, Labels: {y_val.shape}")


## 2. Standard CNN Architecture Definition


In [ ]:
inputs = layers.Input(shape=(224, 224, 3), name="input_image")
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.08)(x)

# 4 Convolutional Feature Extraction Blocks
x = layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv1")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv3")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(256, (3, 3), padding="same", activation="relu", name="conv4_features")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dense(128, activation="relu", name="dense_shared")(x)
x = layers.Dropout(0.35)(x)

# Softmax 3-class Nominal Output
outputs = layers.Dense(3, activation="softmax", name="ripeness_output")(x)

std_model = keras.Model(inputs=inputs, outputs=outputs, name="Standard_CNN")
std_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
std_model.summary()


## 3. Training & Checkpointing


In [ ]:
history = std_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)
std_model.save(MODELS_DIR / "standard_cnn.keras")
print("Saved Model A: standard_cnn.keras")


## 4. Plot Training Accuracy & Loss


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history["accuracy"], label="Train Acc", color="#2a9d8f")
ax1.plot(history.history["val_accuracy"], label="Val Acc", color="#264653", linestyle="--")
ax1.set_title("Standard CNN Accuracy")
ax1.legend()

ax2.plot(history.history["loss"], label="Train Loss", color="#e76f51")
ax2.plot(history.history["val_loss"], label="Val Loss", color="#b03a2e", linestyle="--")
ax2.set_title("Standard CNN Loss")
ax2.legend()
plt.show()
